# Analysis of constant acceleration MPC, for the C++ implementation

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
import functools
import tqdm
import h5py
import numpy as np
import jax
import jax.numpy as jnp

import matplotlib.pyplot as plt

import exp_mpc.stewart_min.mpc_spec as mpc_spec
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.viz as viz

In [ ]:
# general setup (needs to conform with `mpc_export.py`
weights = mpc_spec.ExpWeights(
    lin_dyn=jnp.ones(3) * 1e5,
    omega=jnp.ones(3) * 5e5,
    control=jnp.ones(6) * 1e-1,
    alpha_acc=jnp.array([1.0]),
    alpha_omega=jnp.array([1.0]),
)
limits = mpc_spec.MPCLimits()
spec = mpc_spec.MPCSpec.init_weight_margins(
    weights, limits, max_iter=2, max_ls=2, use_terminal=True
)
acc_ref = jnp.array([1.0, 0.0, 9.81])
omega_ref = jnp.array([0.0, 0.0, 0.1])
acc_ref = jnp.tile(A=acc_ref, reps=(spec.n, 1))
omega_ref = jnp.tile(A=omega_ref, reps=(spec.n, 1))

## visualize results

In [ ]:
with h5py.File("../../cpp/data/mpc_example_data.h5", "r") as f:
    control_res = np.array(f["control_res"])
    timings  = np.array(f["timings"]) * 1e-6

train_state = opt.TrainState.zero_init(spec, mpc_spec.gravity[2])
train_list = []
train_step = jax.jit(functools.partial(opt.apply_control, spec, acc_ref=acc_ref, omega_ref=omega_ref))
for i in tqdm.tqdm(range(control_res.shape[0])):
    train_state = train_step(train_state, control_res[i])
    train_list.append(train_state)

In [ ]:
freqs = 1.0 / np.array(timings)
print(f"{float(np.min(freqs)):.2f}, {float(np.max(freqs)):.2f}, {float(np.mean(freqs)):.2f}, {float(np.std(freqs)):.2f}")

In [ ]:
plt.close("all")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(freqs)
ax.set_title("Optimization Frequency Over Time")
ax.set_xlabel("Iteration")
ax.set_ylabel("Frequency (Hz)")
ax.grid(True)
plt.show()

In [ ]:
sol_list_end = []
extra_steps = 0

# sol_list_end = viz.split_tablesol(sol_list[-1])
# extra_steps = sol_list[-1].u.shape[0] - 1

tl = train_list
references= {
    "xyz-acceleration": jnp.tile(A=acc_ref[0], reps=(len(tl), 1)),
    "angular-velocity": jnp.tile(A=omega_ref[0], reps=(len(tl), 1)),
}

In [ ]:
mpc_human_fig = viz.plot_human_trajectory(trajectory=tl, limits=limits, spec=spec, references=references)

In [ ]:
mpc_vestibular_fig = viz.plot_vestibular_trajectory(trajectory=tl, limits=limits, spec=spec)

In [ ]:
mpc_table_fig = viz.plot_cartesian_table_trajectory(trajectory=tl, limits=limits, spec=spec)

In [ ]:
mpc_actuator_fig = viz.plot_actuator_trajectory(trajectory=tl, limits=limits, spec=spec)

## visualize timings

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
ax.plot(np.array(timings) * 1e3)
fig.suptitle("C++ timings")
ax.set_xlabel("iter")
ax.set_ylabel("time (ms)")
ax.set_ylim(2, 10)
ax.grid()
plt.show()

## animation

(WARNING: can take a long time.
Usually about as long as the video being generated.)

In [ ]:
# mp_mpl.call_mp_animate_trajectory(
#     file_name="data/cpp_const_acc_3d.mp4",
#     trajectory=trajectory,
# )